# Регистрация BGE-M3 в MLflow

Артефакты загружаются через HTTPS MLflow. Presigned multipart upload/download отключены: внутренний адрес rustfs:9000 недоступен с локального компьютера. После изменения настроек повторно выполните первую ячейку перед загрузкой модели.

In [ ]:
import os
from importlib.metadata import version
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from mlflow.models import infer_signature
from sentence_transformers import SentenceTransformer

In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()
)
load_dotenv(PROJECT_ROOT / ".env")
os.environ["MLFLOW_ENABLE_PROXY_MULTIPART_UPLOAD"] = "false"
os.environ["MLFLOW_ENABLE_PROXY_MULTIPART_DOWNLOAD"] = "false"
os.environ["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "1800"
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

MODEL_NAME = "patent-bi-encoder"
MODEL_ID = "BAAI/bge-m3"
MODEL_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
MAX_SEQ_LENGTH = 1024
EMBEDDING_RUN_ID = "59c0674b2f2849bf9fd60f1580aa31dc"
QDRANT_COLLECTION = "patents_bge_m3_v1_59c0674b"
WEIGHTS_DIR = PROJECT_ROOT / "data" / "models" / f"bge-m3-{MODEL_REVISION}"
REQUIREMENTS = [
    f"{package}=={version(package)}"
    for package in (
        "mlflow",
        "sentence-transformers",
        "transformers",
        "torch",
        "numpy",
        "pandas",
        "cloudpickle",
    )
]

## 1. Подготовить пакет модели

In [ ]:
class PatentEncoder(mlflow.pyfunc.PythonModel):
    def load_context(self, context: mlflow.pyfunc.PythonModelContext) -> None:
        self.encoder = SentenceTransformer(
            context.artifacts["weights"],
            device="cpu",
            local_files_only=True,
        )
        self.encoder.max_seq_length = 1024

    def predict(
        self,
        context: mlflow.pyfunc.PythonModelContext,
        model_input: pd.DataFrame,
    ) -> np.ndarray:
        return self.encoder.encode(
            model_input["text"].tolist(),
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )


encoder = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION, device="cpu")
encoder.max_seq_length = MAX_SEQ_LENGTH
WEIGHTS_DIR.parent.mkdir(parents=True, exist_ok=True)
encoder.save(str(WEIGHTS_DIR))

example = pd.DataFrame(
    {
        "text": [
            "Methods for recycling lithium-ion batteries",
            "Optical sensors for autonomous navigation",
        ]
    }
)
expected = encoder.encode(
    example["text"].tolist(),
    normalize_embeddings=True,
    convert_to_numpy=True,
)
signature = infer_signature(example, expected)
del encoder
print(f"Веса сохранены: {WEIGHTS_DIR}")

## 2. Загрузить в MLflow

In [ ]:
mlflow.set_experiment("patent-model-registration")
with mlflow.start_run(run_name="register-bge-m3"):
    mlflow.log_params(
        {
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "max_seq_length": MAX_SEQ_LENGTH,
            "normalize_embeddings": True,
            "embedding_dimension": 1024,
        }
    )
    mlflow.set_tags(
        {
            "embedding_run_id": EMBEDDING_RUN_ID,
            "qdrant_collection": QDRANT_COLLECTION,
            "document_input": "title + newline + abstract",
        }
    )
    model_info = mlflow.pyfunc.log_model(
        name="encoder",
        python_model=PatentEncoder(),
        artifacts={"weights": str(WEIGHTS_DIR)},
        signature=signature,
        input_example=example,
        pip_requirements=REQUIREMENTS,
        metadata={
            "hf_revision": MODEL_REVISION,
            "qdrant_collection": QDRANT_COLLECTION,
            "embedding_run_id": EMBEDDING_RUN_ID,
        },
    )
print(f"Модель в MLflow: {model_info.model_uri}")

## 3. Проверить восстановление и зарегистрировать

In [ ]:
restored = mlflow.pyfunc.load_model(model_info.model_uri)
actual = np.asarray(restored.predict(example))
assert actual.shape == (len(example), 1024)
assert np.allclose(actual, expected, atol=1e-5, rtol=1e-4)
assert np.allclose(np.linalg.norm(actual, axis=1), 1.0, atol=1e-3)
del restored

registered = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=MODEL_NAME,
    tags={
        "qdrant_collection": QDRANT_COLLECTION,
        "embedding_run_id": EMBEDDING_RUN_ID,
        "hf_revision": MODEL_REVISION,
    },
)
print(f"Проверено и зарегистрировано: {MODEL_NAME}, версия {registered.version}")

## 4. Назначить champion

In [ ]:
client = mlflow.MlflowClient()
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=registered.version,
)
champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
assert str(champion.version) == str(registered.version)
print(f"Champion: {MODEL_NAME}, версия {champion.version}")
print(f"Model URI: models:/{MODEL_NAME}@champion")
print(f"Qdrant: {champion.tags['qdrant_collection']}")